In [ ]:
from fastllm import Msg, Part, estimate_cost, ModelPrice, acompletion

OPENAI_API_KEY = os.environ.get('OPENAI_API_KEY')
ANTHROPIC_API_KEY = os.environ.get('ANTHROPIC_API_KEY')
GEMINI_API_KEY = os.environ.get('GEMINI_API_KEY')
MOONSHOT_API_KEY = os.environ.get('MOONSHOT_API_KEY')

assert OPENAI_API_KEY, 'Missing OPENAI_API_KEY'
assert ANTHROPIC_API_KEY, 'Missing ANTHROPIC_API_KEY'
assert GEMINI_API_KEY, 'Missing GEMINI_API_KEY'

OPENAI_MODEL = os.environ.get('FASTLLM_OPENAI_MODEL', 'gpt-5-mini')
OPENAI_EMBED_MODEL = os.environ.get('FASTLLM_OPENAI_EMBED_MODEL', 'text-embedding-3-small')
ANTHROPIC_MODEL = os.environ.get('FASTLLM_ANTHROPIC_MODEL', 'claude-sonnet-4-5')
GEMINI_MODEL = os.environ.get('FASTLLM_GEMINI_MODEL', 'gemini-2.5-flash')
GEMINI_EMBED_MODEL = os.environ.get('FASTLLM_GEMINI_EMBED_MODEL', 'models/gemini-embedding-001')
MOONSHOT_MODEL = os.environ.get('FASTLLM_MOONSHOT_MODEL', 'kimi-k2.5')

RUN_OPTIONAL = os.environ.get('FASTLLM_NB_RUN_OPTIONAL', '0') == '1'
SHOW_ALL_OPS = os.environ.get('FASTLLM_SHOW_ALL_OPS', '0') == '1'

print('models:', OPENAI_MODEL, ANTHROPIC_MODEL, GEMINI_MODEL, MOONSHOT_MODEL)
print('optional sections:', RUN_OPTIONAL)
print('show full op index:', SHOW_ALL_OPS)
print('moonshot key present:', bool(MOONSHOT_API_KEY))


models: gpt-5-mini claude-sonnet-4-5 gemini-2.5-flash kimi-k2.5
optional sections: False
show full op index: False
moonshot key present: True


In [ ]:
import base64
def make_demo_png_b64(w=128, h=128, rgb=(30, 130, 200)):
    import binascii, struct, zlib
    r, g, b = rgb
    row = bytes([r, g, b]) * w
    raw = b''.join(b'\x00' + row for _ in range(h))
    comp = zlib.compress(raw)

    def chunk(tag, data):
        return struct.pack('!I', len(data)) + tag + data + struct.pack('!I', binascii.crc32(tag + data) & 0xffffffff)

    ihdr = struct.pack('!IIBBBBB', w, h, 8, 2, 0, 0, 0)
    png = b'\x89PNG\r\n\x1a\n' + chunk(b'IHDR', ihdr) + chunk(b'IDAT', comp) + chunk(b'IEND', b'')
    return base64.b64encode(png).decode('ascii')


DEMO_PNG_B64 = make_demo_png_b64()

In [ ]:
from lisette.core import lite_mk_func
from fastllm import acollect_stream, afile_create, afile_list, afile_get, afile_content, to_input_file_part

In [ ]:
def simple_add(
    a: int,   # first operand
    b: int=0  # second operand
) -> int:
    "Add two numbers together"
    return a + b

In [ ]:
toolsc = lite_mk_func(simple_add)
toolsc

{'type': 'function',
 'function': {'name': 'simple_add',
  'description': 'Add two numbers together\n\nReturns:\n- type: integer',
  'parameters': {'type': 'object',
   'properties': {'a': {'type': 'integer', 'description': 'first operand'},
    'b': {'type': 'integer', 'description': 'second operand', 'default': 0}},
   'required': ['a']}}}

# High-Level API (acompletion)

## No Stream

In [ ]:
res = await acompletion(model=OPENAI_MODEL, messages=[{'role': 'user', 'content': 'Say hello in one short line.'}])
res.message.content[0].text, res.usage

('Hello!',
 Usage(prompt_tokens=13, completion_tokens=94, total_tokens=107, raw={'input_tokens': 13, 'input_tokens_details': {'cached_tokens': 0}, 'output_tokens': 94, 'output_tokens_details': {'reasoning_tokens': 64}, 'total_tokens': 107}))

In [ ]:
res = await acompletion(model=ANTHROPIC_MODEL, messages=[{'role': 'user', 'content': 'Say hello in one short line.'}])
res.message.content[0].text, res.usage

('Hello! How can I help you today?',
 Usage(prompt_tokens=14, completion_tokens=12, total_tokens=26, raw={'input_tokens': 14, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'cache_creation': {'ephemeral_5m_input_tokens': 0, 'ephemeral_1h_input_tokens': 0}, 'output_tokens': 12, 'service_tier': 'standard', 'inference_geo': 'not_available'}))

In [ ]:
res = await acompletion(model=GEMINI_MODEL, messages=[{'role': 'user', 'content': 'Say hello in one short line.'}])
res.message.content[0].text, res.usage

('Hello!',
 Usage(prompt_tokens=8, completion_tokens=2, total_tokens=36, raw={'promptTokenCount': 8, 'candidatesTokenCount': 2, 'totalTokenCount': 36, 'promptTokensDetails': [{'modality': 'TEXT', 'tokenCount': 8}], 'thoughtsTokenCount': 26}))

In [ ]:
res = await acompletion(model=MOONSHOT_MODEL, messages=[{'role': 'user', 'content': 'Say hello in one short line.'}],  api_key=MOONSHOT_API_KEY, base_url='https://api.moonshot.ai/v1')
res.message.content[0].text, res.usage

('Hello there!',
 Usage(prompt_tokens=14, completion_tokens=86, total_tokens=100, raw={'prompt_tokens': 14, 'completion_tokens': 86, 'total_tokens': 100, 'cached_tokens': 14, 'prompt_tokens_details': {'cached_tokens': 14}}))

## Stream (Rest of the notebook uses `stream=True`)

In [ ]:
res = await acollect_stream(acompletion(model=OPENAI_MODEL, messages=[{'role': 'user', 'content': 'Say hello in one short line.'}], stream=True))

In [ ]:
L(dir(res)).filter(lambda o:not o.startswith('__'))

['deltas', 'final', 'finish_reason', 'raw_events', 'text', 'tool_calls', 'usage']

In [ ]:
res.usage, res.text, res.tool_calls

(Usage(prompt_tokens=13, completion_tokens=79, total_tokens=92, raw={'input_tokens': 13, 'input_tokens_details': {'cached_tokens': 0}, 'output_tokens': 79, 'output_tokens_details': {'reasoning_tokens': 64}, 'total_tokens': 92}),
 'Hello!',
 [])

In [ ]:
res = await acollect_stream(acompletion(model=ANTHROPIC_MODEL, messages=[{'role': 'user', 'content': 'Say hello in one short line.'}], stream=True))

In [ ]:
res.usage, res.text, res.tool_calls

(Usage(prompt_tokens=14, completion_tokens=12, total_tokens=26, raw={'input_tokens': 14, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'output_tokens': 12}),
 'Hello! How can I help you today?',
 [])

In [ ]:
res = await acollect_stream(acompletion(model=GEMINI_MODEL, messages=[{'role': 'user', 'content': 'Say hello in one short line.'}], stream=True))

In [ ]:
res.usage, res.text, res.tool_calls

(Usage(prompt_tokens=8, completion_tokens=2, total_tokens=31, raw={'promptTokenCount': 8, 'candidatesTokenCount': 2, 'totalTokenCount': 31, 'promptTokensDetails': [{'modality': 'TEXT', 'tokenCount': 8}], 'thoughtsTokenCount': 21}),
 'Hello!',
 [])

In [ ]:
res = await acollect_stream(acompletion(model=MOONSHOT_MODEL, messages=[{'role': 'user', 'content': 'Say hello in one short line.'}], stream=True, api_key=MOONSHOT_API_KEY, base_url='https://api.moonshot.ai/v1'))

In [ ]:
res.usage, res.text, res.tool_calls

(Usage(prompt_tokens=14, completion_tokens=185, total_tokens=199, raw={'prompt_tokens': 14, 'completion_tokens': 185, 'total_tokens': 199, 'cached_tokens': 14, 'prompt_tokens_details': {'cached_tokens': 14}}),
 'Hello!',
 [])

## Tool Call

In [ ]:
pr = "What is 5478954793+547982745? How about 5479749754+9875438979? Always use tools for calculations, and describe what you'll do before using a tool. Where multiple tool calls are required, do them in a single response where possible. "

In [ ]:
res = await acollect_stream(acompletion(model=OPENAI_MODEL, messages=[{'role': 'user', 'content': pr}], stream=True, tools=[toolsc]))

In [ ]:
L(dir(res)).filter(lambda o:not o.startswith('__'))

['deltas', 'final', 'finish_reason', 'raw_events', 'text', 'tool_calls', 'usage']

In [ ]:
res.usage, res.text, res.tool_calls

(Usage(prompt_tokens=124, completion_tokens=720, total_tokens=844, raw={'input_tokens': 124, 'input_tokens_details': {'cached_tokens': 0}, 'output_tokens': 720, 'output_tokens_details': {'reasoning_tokens': 576}, 'total_tokens': 844}),
 "I'll compute both sums using the built-in add tool. I'll call the add function twice in parallel (one call per sum) and then report the results. Now I'll run both tool calls.",
 [ToolCall(id='call_o4mhekR9h95AwBLyHRxcrIBs', name='simple_add', arguments={'a': 5478954793, 'b': 547982745}),
  ToolCall(id='call_Q4Sq1km8XbXm4XaVmafR2CO8', name='simple_add', arguments={'a': 5479749754, 'b': 9875438979})])

In [ ]:
res = await acollect_stream(acompletion(model=ANTHROPIC_MODEL, messages=[{'role': 'user', 'content': pr}], stream=True, tools=[toolsc]))

In [ ]:
res.usage, res.text, res.tool_calls

(Usage(prompt_tokens=659, completion_tokens=217, total_tokens=876, raw={'input_tokens': 659, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'output_tokens': 217}),
 "I'll help you calculate both of those sums using the addition tool.\n\nLet me break down what I need to do:\n1. First calculation: 5478954793 + 547982745\n2. Second calculation: 5479749754 + 9875438979\n\nSince these two calculations are independent of each other, I'll perform both additions in a single response.",
 [ToolCall(id='toolu_018eKjrmciV4uTpdwGcqCQNn', name='simple_add', arguments={'a': 5478954793, 'b': 547982745}),
  ToolCall(id='toolu_01HqULdbZMj7tw8rz9p9WKRH', name='simple_add', arguments={'a': 5479749754, 'b': 9875438979})])

In [ ]:
res = await acollect_stream(acompletion(model=GEMINI_MODEL, messages=[{'role': 'user', 'content': pr}], stream=True, tools=[toolsc]))

In [ ]:
res.usage, res.text, res.tool_calls

(Usage(prompt_tokens=149, completion_tokens=96, total_tokens=283, raw={'promptTokenCount': 149, 'candidatesTokenCount': 96, 'totalTokenCount': 283, 'promptTokensDetails': [{'modality': 'TEXT', 'tokenCount': 149}], 'thoughtsTokenCount': 38}),
 'I will use the `simple_add` tool to calculate the sum of the two pairs of numbers.',
 [ToolCall(id='call_0', name='simple_add', arguments={'a': 5478954793, 'b': 547982745}),
  ToolCall(id='call_1', name='simple_add', arguments={'a': 5479749754, 'b': 9875438979})])

In [ ]:
res = await acollect_stream(acompletion(model=MOONSHOT_MODEL, messages=[{'role': 'user', 'content': pr}], stream=True, api_key=MOONSHOT_API_KEY, base_url='https://api.moonshot.ai/v1', tools=[toolsc]))

In [ ]:
res.usage, res.text, res.tool_calls

(Usage(prompt_tokens=134, completion_tokens=233, total_tokens=367, raw={'prompt_tokens': 134, 'completion_tokens': 233, 'total_tokens': 367, 'cached_tokens': 134, 'prompt_tokens_details': {'cached_tokens': 134}}),
 "I'll calculate both sums using the addition tool:\n\n1. First, I'll add 5478954793 + 547982745\n2. Second, I'll add 5479749754 + 9875438979\n\nLet me make both calculations now:",
 [ToolCall(id='simple_add:0', name='simple_add', arguments={'a': 5478954793, 'b': 547982745}),
  ToolCall(id='simple_add:1', name='simple_add', arguments={'a': 5479749754, 'b': 9875438979})])

## Web Search

In [ ]:
pr = "Search the web for weather in istanbul"

In [ ]:
res = await acollect_stream(acompletion(model=OPENAI_MODEL, messages=[{'role': 'user', 'content': pr}], stream=True, tools=[{"type": "web_search"}]))

In [ ]:
res.usage, res.text, res.tool_calls

(Usage(prompt_tokens=5135, completion_tokens=1165, total_tokens=6300, raw={'input_tokens': 5135, 'input_tokens_details': {'cached_tokens': 0}, 'output_tokens': 1165, 'output_tokens_details': {'reasoning_tokens': 768}, 'total_tokens': 6300}),
 "I searched the web — here's the weather for Istanbul, Turkey as of Monday, March 30, 2026.\n\n- Current conditions (now): Cloudy, 48°F (9°C). \n\n- 7-day summary:\n  - Mon, March 30: Cloudy; rain this morning then brief showers this afternoon — High 52°F (11°C), Low 49°F (10°C).\n  - Tue, March 31: Clouds then brightening — High 63°F (17°C), Low 51°F (10°C).\n  - Wed, Apr 1: Mainly cloudy, breezy afternoon — High 65°F (18°C), Low 51°F (10°C).\n  - Thu, Apr 2: A couple of morning showers; clouds and sun otherwise — High 64°F (18°C), Low 51°F (11°C).\n  - Fri, Apr 3: Cloudy with a few afternoon showers — High 61°F (16°C), Low 53°F (12°C).\n  - Sat, Apr 4: Variable clouds with a passing shower or two — High 59°F (15°C), Low 50°F (10°C).\n  - Sun, Ap

Looks like citations are no longer included, and Anthropic uses code execution for web search:

In [ ]:
res = await acollect_stream(acompletion(model=ANTHROPIC_MODEL, messages=[{"role":"user","content": pr}], stream=True,
    tools=[{"type":"web_search_20260209","name":"web_search"}]))

In [ ]:
res.usage, res.text, res.tool_calls

(Usage(prompt_tokens=9486, completion_tokens=456, total_tokens=9942, raw={'input_tokens': 9486, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'output_tokens': 456, 'server_tool_use': {'web_search_requests': 1, 'web_fetch_requests': 0}}),
 'Based on the current weather search results for Istanbul:\n\nThe weather in Istanbul today shows temperatures around +9°C, with conditions described as "Cool".\n\nAccuWeather reports a high of 63°F (about 17°C) with mostly cloudy conditions. The current temperature is around 49°F with winds from the WSW at 9 mph.\n\nSunrise is at 6:51 AM and sunset at 7:26 PM.\n\nThe weather appears to be cool with mostly cloudy skies throughout the day. You can check the links above for more detailed hourly forecasts and extended weather information.',
 [ToolCall(id='0', name='', arguments={'code': '\nimport json\n\nresult = await web_search({"query": "weather Istanbul today"})\nparsed = json.loads(result)\n\n# Print only key information from the s

In [ ]:
res = await acollect_stream(acompletion(model=GEMINI_MODEL, messages=[{'role': 'user', 'content': pr}], stream=True, tools=[{"googleSearch": {}}]))

In [ ]:
res.usage, res.text, res.tool_calls

(Usage(prompt_tokens=8, completion_tokens=496, total_tokens=1128, raw={'promptTokenCount': 8, 'candidatesTokenCount': 496, 'totalTokenCount': 1128, 'promptTokensDetails': [{'modality': 'TEXT', 'tokenCount': 8}], 'toolUsePromptTokenCount': 61, 'toolUsePromptTokensDetails': [{'modality': 'TEXT', 'tokenCount': 61}], 'thoughtsTokenCount': 563}),
 "The current weather in Istanbul, Turkey, is cloudy with a temperature of 10°C (50°F). The chance of rain is approximately 31%, and it feels like 8°C (47°F). The humidity is around 93%. The local time in Istanbul is 08:22 PM.\n\n**Today's Forecast (Monday, March 30, 2026):**\nRain is expected during the day, with a 75% chance, and mostly cloudy conditions at night, with a 40% chance of rain. Temperatures will range between 9°C (48°F) and 11°C (51°F), and the humidity will be around 92%.\n\n**Upcoming Weather Forecast:**\n*   **Tuesday, March 31:** Cloudy during the day with light rain at night. Temperatures are predicted to be between 9°C (49°F) a

In [ ]:
L([nested_idx(o["candidates"][0], 'groundingMetadata', 'groundingSupports') for o in res.raw_events]).filter()

[[{'segment': {'startIndex': 161, 'endIndex': 188, 'text': 'The humidity is around 93%.'}, 'groundingChunkIndices': [0]}, {'segment': {'startIndex': 189, 'endIndex': 228, 'text': 'The local time in Istanbul is 08:22 PM.'}, 'groundingChunkIndices': [1]}, {'segment': {'startIndex': 230, 'endIndex': 394, 'text': "**Today's Forecast (Monday, March 30, 2026):**\nRain is expected during the day, with a 75% chance, and mostly cloudy conditions at night, with a 40% chance of rain."}, 'groundingChunkIndices': [0]}, {'segment': {'startIndex': 395, 'endIndex': 495, 'text': 'Temperatures will range between 9°C (48°F) and 11°C (51°F), and the humidity will be around 92%.'}, 'groundingChunkIndices': [0]}, {'segment': {'startIndex': 603, 'endIndex': 675, 'text': 'Temperatures are predicted to be between 9°C (49°F) and 16°C (60°F).'}, 'groundingChunkIndices': [0, 2]}, {'segment': {'startIndex': 676, 'endIndex': 819, 'text': '*   **Wednesday, April 1:** Light rain is expected throughout the day and nig

In [ ]:
res = await acollect_stream(acompletion(
    model=MOONSHOT_MODEL,
    api_key=MOONSHOT_API_KEY,
    base_url="https://api.moonshot.ai/v1",
    endpoint="chat",
    messages=[{"role":"user","content": pr}],
    stream=True,
    native={
        "tools": [{"type":"builtin_function","function":{"name":"$web_search"}}],
        "tool_choice": "auto",
    },
))

In [ ]:
res.usage, res.text, res.tool_calls, res.finish_reason

(Usage(prompt_tokens=15, completion_tokens=1, total_tokens=16, raw={'prompt_tokens': 15, 'completion_tokens': 1, 'total_tokens': 16}),
 '',
 [ToolCall(id='t-web_search-69cab15502d3', name='$web_search', arguments={'search_result': {'search_id': 'a12994b969cab1554f50a500012334e6'}, 'usage': {'total_tokens': 7803}})],
 'tool_calls')

In [ ]:
import json
from fastllm import acompletion, acollect_stream

native_cfg = {
    "tools": [{"type": "builtin_function", "function": {"name": "$web_search"}}],
    "tool_choice": "auto",
    "thinking": {"type": "disabled"},  # avoids reasoning_content error in follow-up turn
}

messages = [{"role": "user", "content": pr}]

for _ in range(3):
    res = await acollect_stream(acompletion(
        model=MOONSHOT_MODEL,
        api_key=MOONSHOT_API_KEY,
        base_url="https://api.moonshot.ai/v1",
        endpoint="chat",
        messages=messages,
        stream=True,
        native=native_cfg,
    ))

    if res.finish_reason != "tool_calls" or not res.tool_calls:
        print(res.text)
        break

    assistant_tool_calls = [{
        "id": tc.id,
        "type": "function",
        "function": {"name": tc.name, "arguments": json.dumps(tc.arguments)},
    } for tc in res.tool_calls]

    messages.append({"role": "assistant", "content": "", "tool_calls": assistant_tool_calls})
    for tc in res.tool_calls:
        messages.append({
            "role": "tool",
            "tool_call_id": tc.id,
            "content": json.dumps(tc.arguments),
        })

I searched for weather information in Istanbul. Here's what I found:

**Current Weather in Istanbul:**

- **Temperature:** Approximately 11-13°C (52-55°F)
- **Conditions:** Partly cloudy to mostly cloudy
- **Humidity:** Around 70-75%
- **Wind:** Light to moderate winds

**General Forecast:**
- The weather is cool and autumn-like
- Some scattered showers possible
- Nights are cooler, dropping to around 8-10°C (46-50°F)

**What to Expect:**
- It's a good idea to bring a light jacket or sweater
- An umbrella might be useful for occasional rain
- Layered clothing is recommended for the temperature variations between day and night

Would you like more specific information, such as a multi-day forecast or details about a particular time period?


## Thinking

In [ ]:
pr = "Solve 27*43 carefully, then return only the final number."

In [ ]:
res = await acollect_stream(acompletion(model=OPENAI_MODEL, messages=[{'role': 'user', 'content': pr}], stream=True, reasoning_effort='low'))

In [ ]:
res.usage, res.text

(Usage(prompt_tokens=20, completion_tokens=59, total_tokens=79, raw={'input_tokens': 20, 'input_tokens_details': {'cached_tokens': 0}, 'output_tokens': 59, 'output_tokens_details': {'reasoning_tokens': 0}, 'total_tokens': 79}),
 '1161')

In [ ]:
res = await acollect_stream(acompletion(model=OPENAI_MODEL, messages=[{'role': 'user', 'content': pr}], stream=True, reasoning_effort='medium'))

In [ ]:
res.usage, res.text

(Usage(prompt_tokens=20, completion_tokens=74, total_tokens=94, raw={'input_tokens': 20, 'input_tokens_details': {'cached_tokens': 0}, 'output_tokens': 74, 'output_tokens_details': {'reasoning_tokens': 64}, 'total_tokens': 94}),
 '1161')

Provider model default:

In [ ]:
res = await acollect_stream(acompletion(model=ANTHROPIC_MODEL, messages=[{"role":"user","content": pr}], stream=True))

In [ ]:
res.usage, res.text

(Usage(prompt_tokens=23, completion_tokens=69, total_tokens=92, raw={'input_tokens': 23, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'output_tokens': 69}),
 "I'll solve 27 × 43 step by step.\n\n27 × 43\n= 27 × (40 + 3)\n= 27 × 40 + 27 × 3\n= 1080 + 81\n= 1161\n\n1161")

Set a level:

In [ ]:
res = await acollect_stream(acompletion(model=ANTHROPIC_MODEL, messages=[{"role":"user","content": pr}], stream=True, reasoning_effort='medium'))

In [ ]:
res.usage, res.text

(Usage(prompt_tokens=52, completion_tokens=215, total_tokens=267, raw={'input_tokens': 52, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'output_tokens': 215}),
 '1161')

Explicit budget:

In [ ]:
res = await acollect_stream(acompletion(model=ANTHROPIC_MODEL, messages=[{"role":"user","content": pr}], stream=True, reasoning_effort={'type': 'enabled', 'budget_tokens': 1024}))

In [ ]:
res.usage, res.text

(Usage(prompt_tokens=52, completion_tokens=148, total_tokens=200, raw={'input_tokens': 52, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'output_tokens': 148}),
 '1161')

Disable thinking:

In [ ]:
res = await acollect_stream(acompletion(model=GEMINI_MODEL, messages=[{'role': 'user', 'content': pr}], stream=True, reasoning_effort={"thinkingBudget": 0}))

In [ ]:
res.usage, res.text

(Usage(prompt_tokens=17, completion_tokens=168, total_tokens=185, raw={'promptTokenCount': 17, 'candidatesTokenCount': 168, 'totalTokenCount': 185, 'promptTokensDetails': [{'modality': 'TEXT', 'tokenCount': 17}]}),
 'To solve 27 * 43:\n\nWe can break this down:\n27 * 43 = 27 * (40 + 3)\n= (27 * 40) + (27 * 3)\n\nFirst part: 27 * 40\n= 27 * 4 * 10\n= (108) * 10\n= 1080\n\nSecond part: 27 * 3\n= (20 * 3) + (7 * 3)\n= 60 + 21\n= 81\n\nNow add the two parts:\n1080 + 81 = 1161\n\nThe final answer is $\\boxed{1161}$')

Default:

In [ ]:
res = await acollect_stream(acompletion(model=GEMINI_MODEL, messages=[{'role': 'user', 'content': pr}], stream=True))

In [ ]:
res.usage, res.text

(Usage(prompt_tokens=17, completion_tokens=4, total_tokens=337, raw={'promptTokenCount': 17, 'candidatesTokenCount': 4, 'totalTokenCount': 337, 'promptTokensDetails': [{'modality': 'TEXT', 'tokenCount': 17}], 'thoughtsTokenCount': 316}),
 '1161')

Default:

In [ ]:
res = await acollect_stream(acompletion(
    model=MOONSHOT_MODEL,
    api_key=MOONSHOT_API_KEY,
    messages=[{"role":"user","content": pr}],
    stream=True
    # reasoning_effort='low'
))

In [ ]:
res.usage, res.text

(Usage(prompt_tokens=21, completion_tokens=306, total_tokens=327, raw={'prompt_tokens': 21, 'completion_tokens': 306, 'total_tokens': 327, 'cached_tokens': 21, 'prompt_tokens_details': {'cached_tokens': 21}}),
 '1161')

Disabled using API native payload, [official docs](https://platform.moonshot.ai/docs/guide/kimi-k2-5-quickstart#disable-thinking-capability-example):

In [ ]:
res = await acollect_stream(acompletion(
    model=MOONSHOT_MODEL,
    api_key=MOONSHOT_API_KEY,
    messages=[{"role":"user","content": pr}],
    stream=True,
    native={"thinking": {"type": "disabled"}},
))

In [ ]:
res.usage, res.text

(Usage(prompt_tokens=21, completion_tokens=5, total_tokens=26, raw={'prompt_tokens': 21, 'completion_tokens': 5, 'total_tokens': 26}),
 '1161')

## Multimodal

### Images

In [ ]:
parts = [Part(type="text", text="What's in the image?"),
         Part(type="input_image", data={"image_url": f"data:image/png;base64,{DEMO_PNG_B64}"})]
msg1 = Msg('user', parts)
msg2 = Msg('assistant', content=[Part(type="text", text="It's a puppy!")])
msg3 = Msg('user', content=[Part(type="text", text="Try again!")])

In [ ]:
res = await acollect_stream(acompletion(model=OPENAI_MODEL, messages=[msg1,msg2,msg3], stream=True))

In [ ]:
res.usage, res.text

(Usage(prompt_tokens=48, completion_tokens=227, total_tokens=275, raw={'input_tokens': 48, 'input_tokens_details': {'cached_tokens': 0}, 'output_tokens': 227, 'output_tokens_details': {'reasoning_tokens': 128}, 'total_tokens': 275}),
 'The image is a plain, uniform blue square (no discernible objects or details). If you meant to show something else, try uploading a higher-resolution or different file.')

In [ ]:
res = await acollect_stream(acompletion(model=ANTHROPIC_MODEL, messages=[msg1,msg2,msg3], stream=True))

In [ ]:
res.usage, res.text

(Usage(prompt_tokens=56, completion_tokens=45, total_tokens=101, raw={'input_tokens': 56, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'output_tokens': 45}),
 "You're right, my apologies! The image shows a solid blue square or rectangle with a uniform blue color throughout. There's no puppy or any other object in it - just a plain blue background.")

In [ ]:
res = await acollect_stream(acompletion(model=GEMINI_MODEL, messages=[msg1,msg2,msg3], stream=True))

In [ ]:
res.usage, res.text

(Usage(prompt_tokens=277, completion_tokens=24, total_tokens=371, raw={'promptTokenCount': 277, 'candidatesTokenCount': 24, 'totalTokenCount': 371, 'promptTokensDetails': [{'modality': 'TEXT', 'tokenCount': 19}, {'modality': 'IMAGE', 'tokenCount': 258}], 'thoughtsTokenCount': 70}),
 'My apologies! It seems I made a mistake in my previous description.\n\nThe image is a solid color blue rectangle.')

In [ ]:
res = await acollect_stream(acompletion(model=MOONSHOT_MODEL, messages=[msg1,msg2,msg3], stream=True, api_key=MOONSHOT_API_KEY, base_url='https://api.moonshot.ai/v1'))

/Users/keremturgutlu/aai-ws/fastllm/fastllm/clients.py:878: UserWarning: OpenAI-compatible model 'kimi-k2.5' may not fully support canonical media type(s): input_image. If this fails, pass provider-native payloads via `native` or Part.data provider keys.
  payload = self._chat_payload(model, messages, opts, stream=True)


In [ ]:
res.usage, res.text

(Usage(prompt_tokens=58, completion_tokens=181, total_tokens=239, raw={'prompt_tokens': 58, 'completion_tokens': 181, 'total_tokens': 239, 'cached_tokens': 58, 'prompt_tokens_details': {'cached_tokens': 58}}),
 'I see a solid blue square — it appears to be a flat, uniform shade of blue with no distinct objects, patterns, or details visible. It looks like a color swatch or a solid colored background.')

### PDFs

In [ ]:
def b64_file(path): return base64.b64encode(Path(path).read_bytes()).decode()
pdf_fn = '/Users/keremturgutlu/aai-ws/solveit/nbs/samples/solveit.pdf'
pdf_b64 = b64_file(pdf_fn)
parts = [Part(type="text", text="Please summarize this doc in a paragraph"),
         Part(type="input_file", data={"filename": "doc.pdf", "file_data": f"data:application/pdf;base64,{pdf_b64}"})]
msg = Msg('user', parts)

In [ ]:
res = await acollect_stream(acompletion(model=OPENAI_MODEL, messages=[msg], stream=True))

In [ ]:
res.usage, res.text

(Usage(prompt_tokens=1872, completion_tokens=446, total_tokens=2318, raw={'input_tokens': 1872, 'input_tokens_details': {'cached_tokens': 0}, 'output_tokens': 446, 'output_tokens_details': {'reasoning_tokens': 256}, 'total_tokens': 2318}),
 'This one-page overview from fast.ai introduces "How To Solve It With Code," a course by Jeremy Howard that tackles the common problem of getting stuck after the initial burst of productivity when using AI for coding. Howard—co‑founder of fast.ai, creator of its 2016 deep learning certificate, and leader in early LLM and model work—says he discovered a new way to combine AI with coding, tested it with 1,000 students whose results "shocked" him, and is now offering the course more broadly. The curriculum centers on the "solveit method": breaking problems into small, solvable steps with quick iterations and immediate feedback so learners can build, modify, and extend AI-powered projects effectively.')

`filename` is not allowed:

In [ ]:
res = await acollect_stream(acompletion(model=ANTHROPIC_MODEL, messages=[msg], stream=True))

In [ ]:
res.usage, res.text

(Usage(prompt_tokens=1610, completion_tokens=174, total_tokens=1784, raw={'input_tokens': 1610, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'output_tokens': 174}),
 'This document introduces a course called "How To Solve It With Code" by Jeremy Howard from fast.ai. Howard explains that while AI initially makes coding feel effortless, allowing people to build apps in minutes, users often hit a wall when trying to make changes, add features, or fix bugs. Drawing on his extensive experience—including co-founding fast.ai, creating the world\'s first deep learning certificate, and inventing the first LLM—Howard has discovered a new approach to combining AI with coding. After a successful preview with 1,000 students that produced life-changing results, he\'s now opening the course widely. The course teaches the "solveit method," which emphasizes building in small steps with quick iterations and immediate feedback, breaking down complex challenges into completely understan

In [ ]:
res = await acollect_stream(acompletion(model=GEMINI_MODEL, messages=[msg], stream=True))

In [ ]:
res.usage, res.text

(Usage(prompt_tokens=266, completion_tokens=145, total_tokens=1512, raw={'promptTokenCount': 266, 'candidatesTokenCount': 145, 'totalTokenCount': 1512, 'promptTokensDetails': [{'modality': 'TEXT', 'tokenCount': 8}, {'modality': 'DOCUMENT', 'tokenCount': 258}], 'thoughtsTokenCount': 1101}),
 "Jeremy Howard, co-founder of fast.ai, introduces a new course aimed at helping developers overcome the common frustration of getting stuck when using AI for coding after the initial excitement fades. Drawing on fast.ai's mission to democratize AI—a mission supported by its history of creating the first deep learning certificate and inventing the first LLM—Howard has developed a novel method. This approach, termed 'Solveit,' emphasizes building in small, iterative steps with immediate feedback, breaking complex challenges into manageable, solvable pieces. Preview results for the 'Solveit' method were transformative, even for experienced AI practitioners, and it is now being opened to a wider audienc

Supported via `files` api, and here is the [official api docs](https://platform.moonshot.ai/docs/api/files#example-request-1):

In [ ]:
fref = await afile_create(
    model=MOONSHOT_MODEL,
    file=pdf_fn,
    filename="doc.pdf",
    mime_type="application/pdf",
    purpose="file-extract",  # important for docs/pdf extraction
    api_key=MOONSHOT_API_KEY,
    base_url="https://api.moonshot.ai/v1",
)

In [ ]:
fref

FileRef(id='f97d5fcj5z8i11hyejf1', provider='openai_compat', name='', filename='doc.pdf', mime_type='', uri='', size_bytes=845865, raw={'id': 'f97d5fcj5z8i11hyejf1', 'object': 'file', 'bytes': 845865, 'created_at': 1774891441, 'filename': 'doc.pdf', 'purpose': 'file-extract', 'status': 'ok', 'status_details': '', 'file_type': 'application/pdf'})

In [ ]:
files = await afile_list(model=MOONSHOT_MODEL, base_url="https://api.moonshot.ai/v1"); files

[FileRef(id='f97d5fcj5z8i11hyejf1', provider='openai_compat', name='', filename='doc.pdf', mime_type='', uri='', size_bytes=845865, raw={'id': 'f97d5fcj5z8i11hyejf1', 'object': 'file', 'bytes': 845865, 'created_at': 1774891441, 'filename': 'doc.pdf', 'purpose': 'file-extract', 'status': 'ok', 'status_details': '', 'file_type': 'application/pdf'}),
 FileRef(id='f97cs19i3qj111f5zaai', provider='openai_compat', name='', filename='doc.pdf', mime_type='', uri='', size_bytes=845865, raw={'id': 'f97cs19i3qj111f5zaai', 'object': 'file', 'bytes': 845865, 'created_at': 1774890109, 'filename': 'doc.pdf', 'purpose': 'file-extract', 'status': 'ok', 'status_details': '', 'file_type': 'application/pdf'}),
 FileRef(id='f978c7nrdi9i11gw9gzi', provider='openai_compat', name='', filename='doc.pdf', mime_type='', uri='', size_bytes=845865, raw={'id': 'f978c7nrdi9i11gw9gzi', 'object': 'file', 'bytes': 845865, 'created_at': 1774871826, 'filename': 'doc.pdf', 'purpose': 'file-extract', 'status': 'ok', 'statu

In [ ]:
file_part = to_input_file_part(fref); file_part

Part(type='input_file', text=None, data={'file_id': 'f97d5fcj5z8i11hyejf1', 'filename': 'doc.pdf'})

In [ ]:
raw = await afile_content(model=MOONSHOT_MODEL, file_id=fref.id, api_key=MOONSHOT_API_KEY, base_url="https://api.moonshot.ai/v1")
doc_text = raw.decode("utf-8", errors="ignore")

In [ ]:
msg = Msg(role="user", content=[Part(type="text", text=doc_text), Part(type="text", text="Please summarize this doc in a paragraph")])
res = await acollect_stream(acompletion(model=MOONSHOT_MODEL, base_url="https://api.moonshot.ai/v1", messages=[msg], stream=True))
res.usage, res.text

(Usage(prompt_tokens=508, completion_tokens=579, total_tokens=1087, raw={'prompt_tokens': 508, 'completion_tokens': 579, 'total_tokens': 1087, 'cached_tokens': 508, 'prompt_tokens_details': {'cached_tokens': 508}}),
 'Jeremy Howard of fast.ai introduces a new coding course based on the "solve it method," addressing the common frustration where AI-assisted coding initially feels magical but becomes difficult when modifying, debugging, or expanding projects. Drawing from his background democratizing AI—including creating the world\'s first deep learning certificate in 2016 and inventing the first LLM—Howard developed this iterative approach after teaching thousands of students, emphasizing small steps with immediate feedback to break complex challenges into manageable pieces. Following a transformative preview with 1,000 students (including elite AI practitioners), the course is now publicly available to help developers overcome the typical limitations of AI coding tools.')

### Audio

In [ ]:
AUDIO_URL = "https://openaiassets.blob.core.windows.net/$web/API/docs/audio/alloy.wav"
# Download sample audio + encode base64
wav_data = httpx.get(AUDIO_URL, timeout=30).content
wav_b64 = base64.b64encode(wav_data).decode("ascii")

In [ ]:
parts = [Part(type="text", text="What is this audio saying?"),
         Part(type="input_audio", data={"input_audio": {"data": wav_b64, "format": "wav"}})]
msg = Msg(role="user", content=parts)

Responses API doesn't support audio yet, [official docs](https://developers.openai.com/api/docs/guides/migrate-to-responses#responses-benefits)

In [ ]:
try: res = await acollect_stream(acompletion(model=OPENAI_MODEL, messages=[msg], stream=True))
except Exception as e: print(e)

openai responses.stream model=gpt-5-mini status=400 type=invalid_request_error code=invalid_value request_id=req_9b119cee5650434ea216bc8b77b90b45: Invalid value: 'input_audio'. Supported values are: 'input_text', 'input_image', 'output_text', 'refusal', 'input_file', 'computer_screenshot', and 'summary_text'.


In [ ]:
res = await acollect_stream(acompletion(model=GEMINI_MODEL, messages=[msg], stream=True))
res.usage, res.text

(Usage(prompt_tokens=230, completion_tokens=25, total_tokens=653, raw={'promptTokenCount': 230, 'candidatesTokenCount': 25, 'totalTokenCount': 653, 'promptTokensDetails': [{'modality': 'TEXT', 'tokenCount': 7}, {'modality': 'AUDIO', 'tokenCount': 223}], 'thoughtsTokenCount': 398}),
 'The sun rises in the east and sets in the west. This simple fact has been observed by humans for thousands of years.')

### Video

In [ ]:
VIDEO_URL = "https://storage.googleapis.com/github-repo/img/gemini/multimodality_usecases_overview/pixel8.mp4"

In [ ]:
# Download sample video + encode base64 data URL
vid_data = httpx.get(VIDEO_URL, timeout=60).content
vid_b64 = base64.b64encode(vid_data).decode("ascii")
vid_data_url = f"data:video/mp4;base64,{vid_b64}"

In [ ]:
parts = [Part(type="text", text="What's in this video?"),
         Part(type="input_video", data={"video_url": {"url": vid_data_url}})]
msg = Msg(role="user", content=parts)

In [ ]:
res = await acollect_stream(acompletion(model=GEMINI_MODEL, messages=[msg], stream=True))
res.usage, res.text

(Usage(prompt_tokens=17398, completion_tokens=398, total_tokens=19555, raw={'promptTokenCount': 17398, 'candidatesTokenCount': 398, 'totalTokenCount': 19555, 'promptTokensDetails': [{'modality': 'TEXT', 'tokenCount': 8}, {'modality': 'VIDEO', 'tokenCount': 15517}, {'modality': 'AUDIO', 'tokenCount': 1873}], 'thoughtsTokenCount': 1759}),
 'This video showcases a photographer named Saeka Shimada exploring Tokyo at night and demonstrating the low-light video capabilities of the new Google Pixel phone, specifically its "Video Boost" feature with "Night Sight."\n\nHere\'s a breakdown:\n\n*   **0:00-0:02**: Opens with an aerial night shot of a Tokyo street with a train passing on an elevated track. Saeka Shimada, a Tokyo photographer, is introduced.\n*   **0:02-0:12**: Saeka talks about how Tokyo has many different faces, and how the city at night is completely different from the daytime. She is seen walking through dimly lit, narrow alleyways filled with small shops and lanterns.\n*   **0:1

## Toolloop

In [ ]:
import json

In [ ]:
pr = "What is 5478954793+547982745? How about 5479749754+9875438979? Do them in parallel. Always use tools for calculations."

In [ ]:
def _tool_args(tc):
    args = dict(tc.arguments or {})
    if "_raw" in args:
        try: args = json.loads(args["_raw"])
        except Exception: args = {}
    return args

def tool_msg(tc):
    args = _tool_args(tc)
    out = simple_add(int(args["a"]), int(args["b"]))
    return Msg(
        role="tool",
        content=[Part(type="text", text=str(out))],
        data={"tool_call_id": tc.id, "name": tc.name},
    )

OpenAI toolloop (2 turns)

In [ ]:
msgs = [Msg(role="user", content=[Part(type="text", text=pr)])]

res1 = await acollect_stream(acompletion(
    model=OPENAI_MODEL,
    messages=msgs,
    stream=True,
    tools=[toolsc],
))
msgs.append(res1)
msgs.extend([tool_msg(tc) for tc in res1.tool_calls])

res2 = await acollect_stream(acompletion(
    model=OPENAI_MODEL,
    messages=msgs,
    stream=True,
    tools=[toolsc],
))

res1.tool_calls, res2.text, res2.usage

([ToolCall(id='call_Cp6cihAbAvc1I4CjIDmZ4bvP', name='simple_add', arguments={'a': 5478954793, 'b': 547982745}),
  ToolCall(id='call_rlPuORBrFGUgEmMHq3uzqZ0m', name='simple_add', arguments={'a': 5479749754, 'b': 9875438979})],
 'I computed both sums in parallel using the tool:\n\n- 5478954793 + 547982745 = 6,026,937,538\n- 5479749754 + 9,875,438,979 = 15,355,188,733',
 Usage(prompt_tokens=181, completion_tokens=59, total_tokens=240, raw={'input_tokens': 181, 'input_tokens_details': {'cached_tokens': 0}, 'output_tokens': 59, 'output_tokens_details': {'reasoning_tokens': 0}, 'total_tokens': 240}))

Anthropic

In [ ]:
msgs = [Msg(role="user", content=[Part(type="text", text=pr)])]

res1 = await acollect_stream(acompletion(
    model=ANTHROPIC_MODEL,
    messages=msgs,
    stream=True,
    tools=[toolsc],
))
msgs.append(res1)
msgs.extend([tool_msg(tc) for tc in res1.tool_calls])

res2 = await acollect_stream(acompletion(
    model=ANTHROPIC_MODEL,
    messages=msgs,
    stream=True,
    tools=[toolsc],
))

res1.tool_calls, res2.text, res2.usage

([ToolCall(id='toolu_019mh7VQchdsSk3G6tpEsBWF', name='simple_add', arguments={'a': 5478954793, 'b': 547982745}),
  ToolCall(id='toolu_01UcSddsCmrtUXZKNd7s9rj7', name='simple_add', arguments={'a': 5479749754, 'b': 9875438979})],
 'The results are:\n- 5478954793 + 547982745 = **6,026,937,538**\n- 5479749754 + 9875438979 = **15,355,188,733**',
 Usage(prompt_tokens=852, completion_tokens=56, total_tokens=908, raw={'input_tokens': 852, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'output_tokens': 56}))

Gemini

In [ ]:
msgs = [Msg(role="user", content=[Part(type="text", text=pr)])]

res1 = await acollect_stream(acompletion(
    model=GEMINI_MODEL,
    messages=msgs,
    stream=True,
    tools=[toolsc],
))
msgs.append(res1)
msgs.extend([tool_msg(tc) for tc in res1.tool_calls])

res2 = await acollect_stream(acompletion(
    model=GEMINI_MODEL,
    messages=msgs,
    stream=True,
    tools=[toolsc],
))

res1.tool_calls, res2.text, res2.usage

([ToolCall(id='call_0', name='simple_add', arguments={'b': 547982745, 'a': 5478954793}),
  ToolCall(id='call_1', name='simple_add', arguments={'a': 5479749754, 'b': 9875438979})],
 'The sum of 5478954793 and 547982745 is 6026937538. The sum of 5479749754 and 9875438979 is 15355188733.',
 Usage(prompt_tokens=248, completion_tokens=78, total_tokens=326, raw={'promptTokenCount': 248, 'candidatesTokenCount': 78, 'totalTokenCount': 326, 'promptTokensDetails': [{'modality': 'TEXT', 'tokenCount': 248}]}))

Openai Chat

In [ ]:
msgs = [Msg(role="user", content=[Part(type="text", text=pr)])]

res1 = await acollect_stream(acompletion(
    model=MOONSHOT_MODEL,
    base_url="https://api.moonshot.ai/v1",
    messages=msgs,
    stream=True,
    tools=[toolsc],
))
msgs.append(res1)
msgs.extend([tool_msg(tc) for tc in res1.tool_calls])

res2 = await acollect_stream(acompletion(
    model=MOONSHOT_MODEL,
    base_url="https://api.moonshot.ai/v1",
    messages=msgs,
    stream=True,
    tools=[toolsc],
))

res1.tool_calls, res2.text, res2.usage

([ToolCall(id='simple_add:0', name='simple_add', arguments={'a': 5478954793, 'b': 547982745}),
  ToolCall(id='simple_add:1', name='simple_add', arguments={'a': 5479749754, 'b': 9875438979})],
 'Here are the results:\n\n**5478954793 + 547982745 = 6,026,937,538**\n\n**5479749754 + 9875438979 = 15,355,188,733**',
 Usage(prompt_tokens=326, completion_tokens=65, total_tokens=391, raw={'prompt_tokens': 326, 'completion_tokens': 65, 'total_tokens': 391}))